In [1]:
import argparse
import os
import pathlib
import sys
import uuid

import duckdb
import pandas as pd
from cytotable import convert, presets
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from parsl.config import Config
from parsl.executors import HighThroughputExecutor

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)
profile_base_dir = root_dir

In [2]:
if not in_notebook:
    args = parse_args()
    well_fov = args["well_fov"]
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0014_T1"
    well_fov = "C6-2"
    image_based_profiles_subparent_name = "image_based_profiles"

In [3]:
input_sqlite_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/{well_fov}.duckdb"
).resolve(strict=True)
destination_sc_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/sc_profiles_{well_fov}.parquet"
).resolve()
destination_organoid_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/organoid_profiles_{well_fov}.parquet"
).resolve()
destination_nucleocentric_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/nucleocentric_profiles_{well_fov}.parquet"
).resolve()
destination_sc_parquet_file.parent.mkdir(parents=True, exist_ok=True)
dest_datatype = "parquet"

FileNotFoundError: [Errno 2] No such file or directory: '/home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/data/NF0014_T1/image_based_profiles/0.converted_profiles/C6-2/C6-2.duckdb'

In [4]:
# show the tables
with duckdb.connect(input_sqlite_file) as con:
    tables = con.execute("SHOW TABLES").fetchdf()
    print(tables)
    nuclei_table = con.sql("SELECT * FROM Nuclei").df()
    cells_table = con.sql("SELECT * FROM Cell").df()
    cytoplasm_table = con.sql("SELECT * FROM Cytoplasm").df()
    organoid_table = con.sql("SELECT * FROM Organoid").df()
    nucleocentric_table = con.sql("SELECT * FROM Nucleocentric").df()

NameError: name 'input_sqlite_file' is not defined

In [5]:
nuclei_id_set = set(nuclei_table["object_id"].to_list())
cells_id_set = set(cells_table["object_id"].to_list())
cytoplasm_id_set = set(cytoplasm_table["object_id"].to_list())
# find the intersection of the three sets
intersection_set = nuclei_id_set.intersection(cells_id_set, cytoplasm_id_set)
# keep only the rows in the three tables that are in the intersection set
nuclei_table = nuclei_table[nuclei_table["object_id"].isin(intersection_set)]
cells_table = cells_table[cells_table["object_id"].isin(intersection_set)]
cytoplasm_table = cytoplasm_table[cytoplasm_table["object_id"].isin(intersection_set)]

In [6]:
# connect to DuckDB and register the tables
with duckdb.connect() as con:
    con.register("nuclei", nuclei_table)
    con.register("cells", cells_table)
    con.register("cytoplasm", cytoplasm_table)
    # Merge them with SQL
    merged_df = con.execute("""
        SELECT *
        FROM nuclei
        LEFT JOIN cells USING (object_id)
        LEFT JOIN cytoplasm USING (object_id)
    """).df()

## Reorder object IDs

In [7]:
# replace the object_id with a new unique ID
organoid_table["object_id"] = [i for i in range(1, organoid_table.shape[0] + 1)]
merged_df["object_id"] = [i for i in range(1, merged_df.shape[0] + 1)]
nucleocentric_table["object_id"] = [
    i for i in range(1, nucleocentric_table.shape[0] + 1)
]

In [8]:
# save the organoid data as parquet
print(f"Final organoid data shape: {organoid_table.shape}")
organoid_table.to_parquet(destination_organoid_parquet_file, index=False)
organoid_table.head()

Final organoid data shape: (1, 3961)


,object_id,image_set,Organoid_NoChannel_AreaSizeShape_Volume,Organoid_NoChannel_AreaSizeShape_CenterX,Organoid_NoChannel_AreaSizeShape_CenterY,Organoid_NoChannel_AreaSizeShape_CenterZ,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_MinX,Organoid_NoChannel_AreaSizeShape_MaxX,Organoid_NoChannel_AreaSizeShape_MinY,...,Organoid_Mito_Texture_Variance-3-03-256,Organoid_Mito_Texture_Variance-3-04-256,Organoid_Mito_Texture_Variance-3-05-256,Organoid_Mito_Texture_Variance-3-06-256,Organoid_Mito_Texture_Variance-3-07-256,Organoid_Mito_Texture_Variance-3-08-256,Organoid_Mito_Texture_Variance-3-09-256,Organoid_Mito_Texture_Variance-3-10-256,Organoid_Mito_Texture_Variance-3-11-256,Organoid_Mito_Texture_Variance-3-12-256
0,1,C10-1,8323552.0,695.398726,908.387099,21.825544,11686500.0,449,979,660,...,837.461485,851.838893,838.561914,847.985456,835.907707,836.400088,835.762355,848.355497,833.307533,833.548822


In [9]:
print(f"Final merged single cell dataframe shape: {merged_df.shape}")
# save the sc data as parquet
merged_df.to_parquet(destination_sc_parquet_file, index=False)
merged_df.head()

Final merged single cell dataframe shape: (13, 11883)


,object_id,image_set,Nuclei_NoChannel_AreaSizeShape_Volume,Nuclei_NoChannel_AreaSizeShape_CenterX,Nuclei_NoChannel_AreaSizeShape_CenterY,Nuclei_NoChannel_AreaSizeShape_CenterZ,Nuclei_NoChannel_AreaSizeShape_BboxVolume,Nuclei_NoChannel_AreaSizeShape_MinX,Nuclei_NoChannel_AreaSizeShape_MaxX,Nuclei_NoChannel_AreaSizeShape_MinY,...,Cytoplasm_DNA_Texture_Variance-3-03-256,Cytoplasm_DNA_Texture_Variance-3-04-256,Cytoplasm_DNA_Texture_Variance-3-05-256,Cytoplasm_DNA_Texture_Variance-3-06-256,Cytoplasm_DNA_Texture_Variance-3-07-256,Cytoplasm_DNA_Texture_Variance-3-08-256,Cytoplasm_DNA_Texture_Variance-3-09-256,Cytoplasm_DNA_Texture_Variance-3-10-256,Cytoplasm_DNA_Texture_Variance-3-11-256,Cytoplasm_DNA_Texture_Variance-3-12-256
0,1,C10-1,4077.0,248.521707,742.226637,1.000000,5244.0,230,268,720,...,7.748604e-304,7.748604e-304,7.748604e-304,7.748604e-304,7.748604e-304,7.748604e-304,7.748604e-304,7.748604e-304,7.748604e-304,0.0
1,2,C10-1,118749.0,605.367405,796.971579,8.613917,259920.0,522,674,736,...,7.748604e-304,7.748604e-304,7.748604e-304,7.748604e-304,7.748604e-304,7.748604e-304,7.748604e-304,7.748604e-304,7.748604e-304,0.0
2,3,C10-1,25340.0,578.229242,787.458761,17.628414,75710.0,522,656,736,...,7.748604e-304,7.748604e-304,7.748604e-304,7.748604e-304,7.748604e-304,7.748604e-304,7.748604e-304,7.748604e-304,7.748604e-304,0.0
3,4,C10-1,106061.0,1302.791035,1094.818746,6.935132,164268.0,1246,1363,1044,...,7.748604e-304,7.748604e-304,2.130317e-314,7.748604e-304,8.321552e-317,7.748604e-304,7.748604e-304,7.748604e-304,7.748604e-304,0.0
4,5,C10-1,126753.0,569.161266,925.857045,9.330722,228105.0,516,627,860,...,7.748604e-304,7.748604e-304,0.000000e+00,7.748604e-304,0.000000e+00,7.748604e-304,7.748604e-304,7.748604e-304,7.748604e-304,0.0


In [10]:
print(f"Final nucleocentric dataframe shape: {nucleocentric_table.shape}")
# save the nucleocentric data as parquet
nucleocentric_table.to_parquet(destination_nucleocentric_parquet_file, index=False)
nucleocentric_table.head()

Final nucleocentric dataframe shape: (13, 3074)


,object_id,image_set,Nucleocentric_Mito_CHAMMI75_Feature0,Nucleocentric_Mito_CHAMMI75_Feature1,Nucleocentric_Mito_CHAMMI75_Feature10,Nucleocentric_Mito_CHAMMI75_Feature100,Nucleocentric_Mito_CHAMMI75_Feature101,Nucleocentric_Mito_CHAMMI75_Feature102,Nucleocentric_Mito_CHAMMI75_Feature103,Nucleocentric_Mito_CHAMMI75_Feature104,...,Nucleocentric_DNA_SAMMed3D_Feature90,Nucleocentric_DNA_SAMMed3D_Feature91,Nucleocentric_DNA_SAMMed3D_Feature92,Nucleocentric_DNA_SAMMed3D_Feature93,Nucleocentric_DNA_SAMMed3D_Feature94,Nucleocentric_DNA_SAMMed3D_Feature95,Nucleocentric_DNA_SAMMed3D_Feature96,Nucleocentric_DNA_SAMMed3D_Feature97,Nucleocentric_DNA_SAMMed3D_Feature98,Nucleocentric_DNA_SAMMed3D_Feature99
0,1,C10-1,2.554311,-2.499211,6.270025,1.855958,2.479886,-2.476496,-2.470967,0.746163,...,-0.006731,-0.125312,0.050814,-0.010892,0.035865,-0.024403,0.106305,0.254916,0.266566,0.139579
1,2,C10-1,7.105213,0.659223,1.042990,-3.127599,1.111077,1.992849,0.941766,1.696154,...,-0.007641,-0.055770,0.022011,-0.010457,0.021449,0.003285,-0.054165,0.223486,0.396483,0.231863
2,3,C10-1,4.893235,0.385423,2.861571,-1.543698,2.498050,2.142777,-0.935785,-0.771587,...,-0.006493,-0.049796,0.002883,-0.010533,0.031659,-0.033706,-0.029345,0.212940,0.351991,0.266070
3,4,C10-1,0.144605,-4.757784,7.886053,1.660694,1.476611,-5.204203,0.042312,4.300991,...,-0.006590,-0.090895,0.086051,-0.010847,0.027819,0.016349,-0.029114,0.257279,0.364449,0.183671
4,5,C10-1,0.903223,-0.710765,2.194714,1.150269,0.404250,0.147449,1.135373,-0.689170,...,-0.008157,-0.086117,0.021421,-0.010490,-0.000571,-0.032310,-0.092488,0.232491,0.379522,0.152363
